# Допуск: Terminal-Bench: Benchmarking Agents on Hard, Realistic Tasks in Command Line Interfaces

**Статья:** M. A. Merrill и др. — [arXiv:2601.11868v1](https://arxiv.org/abs/2601.11868v1),
17.01.2026. Задания относятся к **Terminal-Bench 2.0** из этой версии статьи.

5 заданий. Под заголовком каждого указан его тип:

- **что есть что** — сопоставьте проверки с понятиями статьи;
- **расхождение со статьёй** — найдите одно отклонение от протокола;
- **перепутаны блоки** — найдите один блок, который выполняется не на своём месте;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду.

Для отклонения укажите место, объясните, что не так, приведите якорь в статье
и опишите исправление словами. Писать, запускать и отлаживать код не нужно.
Вопросы относятся только к заданиям без отклонений.

Общие импорты:

In [ ]:
import torch

Вспомогательные функции считайте корректными. Код моделирует отдельные правила
бенчмарка; это не полный Harbor. Сценарии независимы друг от друга.

## Задание 1. Проверить саму задачу

**Тип задания:** что есть что — код корректен, определите смысл проверок.

Сопоставьте `check_a`, `check_b`, `check_c` с **Specificity**, **Solvability**,
**Integrity** (§2.3, рис. 3, прил. B).
Для `check_a` рассмотрен небольшой полностью перечисленный набор конечных состояний:
булевы векторы `acceptable`, `tests_passed` отмечают допустимость каждого состояния
по инструкции и вердикт тестов. `oracle_tests` — результаты тестов после oracle.
В `check_c` пары булевых векторов относятся к проверенным попыткам обхода: является
ли результат настоящим выполнением задачи и прошёл ли он тесты.

In [ ]:
def check_a(acceptable, tests_passed):
    return bool(torch.equal(acceptable, tests_passed))

def check_b(oracle_tests):
    return bool(oracle_tests.all())

def check_c(valid_completion, tests_passed):
    shortcuts = tests_passed & ~valid_completion
    return not bool(shortcuts.any())

**Вопросы.**

1. Oracle проходит, но один неправильный результат также проходит тесты.
   Какое свойство этим подтверждено и какое нарушено? Почему нужен отдельный
   dummy/no-op агент (§2.3, рис. 3)?
2. Почему успех всех трёх проверок на конечной выборке не доказывает отсутствие
   любого возможного обхода? Какую дополнительную роль играют adversarial exploit
   agent и человек-аудитор (§2.3, прил. B.4, §5)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Чем измерить выполнение

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

`trace.commands` — фактическая последовательность команд агента;
`task.oracle_commands` — команды эталонного решения. `task.tests` проверяют свойства
конечного состояния `environment`. `run_tests(environment, task.tests)` возвращает
булев вектор результатов. Найдите одно отклонение (§2.1, рис. 2).

In [ ]:
def grade_trial(task, environment, trace, run_tests):
    matches = trace.commands == task.oracle_commands
    return float(matches)

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Когда появляются тесты

**Тип задания:** перепутаны блоки — один блок выполняется не на своём месте.

Начальный контейнер из `task.image` не содержит тестов и oracle.
`public_files` — только файлы рабочего окружения, необходимые по инструкции.
Агент имеет доступ к файловой системе контейнера; после возврата из `run_trial`
отдельный оценщик использует файлы из `/tests`. Код оценивания здесь не показан.
Найдите один блок, который нужно перенести
(§2.1, рис. 2; прил. B.2: Dockerfile Reference Check, Tests or Solution in Image).

In [ ]:
def run_trial(task, agent, create_environment):
    environment = create_environment(task.image)
    environment.copy(task.public_files, "/app")
    environment.copy(task.tests, "/tests")
    agent.solve(task.instruction, environment, timeout=task.time_limit)
    return environment

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Что сравнивает лидерборд

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Выборка из табл. 2: строки — GPT-5.2, GPT-5; столбцы — Codex CLI, Terminus 2.
Числа — resolution rate в процентах. Найдите один неверный комментарий
(§3.1–3.4, §4, рис. 1, прил. A.1).

In [ ]:
def compare_scaffolds():
    rates = torch.tensor([[62.9, 54.0],
                          [49.6, 35.2]])                  # (2, 2), percent
    scaffold_gaps = rates[:, 0] - rates[:, 1]             # percentage points
    best_per_model = rates.max(dim=1).values
    # Choosing each model's best scaffold isolates the model's intrinsic ability.
    return scaffold_gaps, best_per_model

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Пять попыток

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Одну фиксированную пару «агент + модель» проверили пять раз на каждой задаче.
`success` — булева матрица `(N, 5)` с независимыми вердиктами задач;
все задачи имеют одинаковое число повторов. Нужна средняя доля решённых задач
за попытку в протоколе статьи. Найдите одно отклонение (§3, §4, прил. A.1–A.3).

In [ ]:
def resolution_rate(success):
    solved = success.any(dim=1)                          # (N,)
    return 100 * solved.float().mean().item()

**Ваш ответ:**

_Напишите ответ здесь._